# Video Activity Recognition (Research Style)
Clean PyTorch pipeline for video classification.

In [2]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader, Subset
from tqdm import tqdm
from PIL import Image

In [3]:
DATASET_PATH = "/content/UCF101"
IMG_SIZE = 128
FRAME_COUNT = 16
BATCH_SIZE = 4
EPOCHS = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [4]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

In [5]:
class VideoDataset(Dataset):

    def __init__(self, root):
        self.samples = []
        self.classes = sorted(os.listdir(root))

        for label, cls in enumerate(self.classes):
            p = os.path.join(root, cls)
            for v in os.listdir(p):
                self.samples.append((os.path.join(p, v), label))

    def load_video(self, path):

        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)

            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return None

        idxs = np.linspace(0, len(frames)-1, FRAME_COUNT).astype(int)
        frames = [frames[i] for i in idxs]

        frames = [transform(frame) for frame in frames]

        return torch.stack(frames)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]
        vid = self.load_video(path)

        if vid is None:
            return self.__getitem__((idx+1) % len(self.samples))

        return vid, label

In [6]:
DATASET_PATH = "UCF-101"
FRAME_COUNT=16
IMG_SIZE=224
BATCH_SIZE=2
EPOCHS=5

In [7]:
!wget --no-check-certificate https://www.crcv.ucf.edu/data/UCF101/UCF101.rar

--2026-04-02 19:22:54--  https://www.crcv.ucf.edu/data/UCF101/UCF101.rar
Resolving www.crcv.ucf.edu (www.crcv.ucf.edu)... 132.170.214.127
Connecting to www.crcv.ucf.edu (www.crcv.ucf.edu)|132.170.214.127|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 6932971618 (6.5G) [application/x-rar-compressed]
Saving to: ‘UCF101.rar’

UCF101.rar          100%[===================>]   6.46G  60.5MB/s    in 75s     

2026-04-02 19:24:09 (88.5 MB/s) - ‘UCF101.rar’ saved [6932971618/6932971618]



In [8]:
!apt-get install unrar -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.


In [9]:
!unrar x UCF101.rar

Streaming output truncated to the last 5000 lines.
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g06_c05.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g06_c06.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g06_c07.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c01.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c02.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c03.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c04.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c05.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c06.avi             62%  OK 
Extracting  UCF-101/PlayingGuitar/v_PlayingGuitar_g07_c07.avi             62%  OK 
Extracting  UCF-101/PlayingGu

In [10]:
DATASET_PATH = "/content/UCF-101"


In [11]:
dataset = VideoDataset(DATASET_PATH)

In [12]:
import random

indices = random.sample(range(len(dataset)), 200)
dataset = Subset(dataset, indices)

In [13]:
num_classes = len(dataset.dataset.classes)

In [14]:
train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

In [15]:
import torchvision.models as models

class VideoModel(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Identity()

        self.lstm = nn.LSTM(512, 256, batch_first=True)

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):

        B,T,C,H,W = x.shape

        x = x.view(B*T, C, H, W)

        features = self.cnn(x)

        features = features.view(B, T, 512)

        out,_ = self.lstm(features)

        out = out[:,-1]

        return self.fc(out)

In [16]:
model = VideoModel(num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 110MB/s]


In [17]:
for epoch in range(EPOCHS):

    model.train()
    running = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for vids, labels in loop:

        vids = vids.to(device)
        labels = labels.to(device)

        out = model(vids)

        loss = criterion(out, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item()

        pred = torch.argmax(out, dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item(), acc=correct/total)

    print("Epoch Loss:", running/len(train_loader))
    print("Accuracy:", correct/total)

Epoch 1/5: 100%|██████████| 100/100 [05:36<00:00,  3.37s/it, acc=0.025, loss=4.9]


Epoch Loss: 4.615237188339234
Accuracy: 0.025


Epoch 2/5: 100%|██████████| 100/100 [05:42<00:00,  3.42s/it, acc=0.125, loss=4.74]


Epoch Loss: 4.209621455669403
Accuracy: 0.125


Epoch 3/5: 100%|██████████| 100/100 [05:28<00:00,  3.28s/it, acc=0.315, loss=4.27]


Epoch Loss: 3.7858331632614135
Accuracy: 0.315


Epoch 4/5: 100%|██████████| 100/100 [05:35<00:00,  3.36s/it, acc=0.44, loss=3.4]


Epoch Loss: 3.3206766963005068
Accuracy: 0.44


Epoch 5/5: 100%|██████████| 100/100 [05:34<00:00,  3.34s/it, acc=0.66, loss=3.04]

Epoch Loss: 2.8291680204868315
Accuracy: 0.66


In [18]:
torch.save(model.state_dict(), "video_model.pth")

In [20]:
class_names = dataset.dataset.classes

In [21]:
def predict_video(video_path):

    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)

        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        print("Error: No frames found")
        return

    # Sample frames
    idxs = np.linspace(0, len(frames)-1, FRAME_COUNT).astype(int)
    frames = [frames[i] for i in idxs]

    # Apply transform
    frames = [transform(frame) for frame in frames]

    video = torch.stack(frames).unsqueeze(0).to(device)  # shape: (1,T,C,H,W)

    with torch.no_grad():
        output = model(video)
        pred = torch.argmax(output, dim=1).item()

    print("Predicted Action:", class_names[pred])

In [22]:
from google.colab import files
uploaded = files.upload()

Saving your_video.mp4.mp4 to your_video.mp4.mp4


In [24]:
predict_video("your_video.mp4.mp4")

Predicted Action: PlayingDaf
